# Projet ML Immobilier — Maine-et-Loire (Dept. 49)

## Prediction des prix immobiliers 2026

**Equipe** : Arthur, Mandengue, Moneli  
**Donnees** : DVF open data — 64 000 transactions reelles (2024 + 2025)  
**Objectif** : Predire l'evolution des prix par type de bien pour 2026

---

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import sys

sys.path.insert(0, os.getcwd())
import config

print("Tout est charge")

Tout est charge


## 1. Les donnees

Transactions immobilieres reelles du departement 49 issues du DVF (Demandes de Valeurs Foncieres).

In [3]:
df = pd.read_csv(config.DATA_CLEAN)

print(f"Nombre de transactions : {len(df)}")
print(f"Types de biens : {df[config.COL_TYPE].value_counts().to_dict()}")
print(f"Annees : {sorted(df['annee'].unique())}")
print(f"Colonnes : {list(df.columns)}")

Nombre de transactions : 9330
Types de biens : {'Maison': 6874, 'Appartement': 2456}
Annees : [np.int64(2024)]
Colonnes : ['id_mutation', 'date_mutation', 'numero_disposition', 'nature_mutation', 'valeur_fonciere', 'adresse_numero', 'adresse_suffixe', 'adresse_nom_voie', 'adresse_code_voie', 'code_postal', 'code_commune', 'nom_commune', 'code_departement', 'ancien_code_commune', 'ancien_nom_commune', 'id_parcelle', 'ancien_id_parcelle', 'numero_volume', 'lot1_numero', 'lot1_surface_carrez', 'lot2_numero', 'lot2_surface_carrez', 'lot3_numero', 'lot3_surface_carrez', 'lot4_numero', 'lot4_surface_carrez', 'lot5_numero', 'lot5_surface_carrez', 'nombre_lots', 'code_type_local', 'type_local', 'surface_reelle_bati', 'nombre_pieces_principales', 'code_nature_culture', 'nature_culture', 'code_nature_culture_speciale', 'nature_culture_speciale', 'surface_terrain', 'longitude', 'latitude', '2025-239276', '2025-01-03', '000001', 'Vente', '160000', '142', 'Unnamed: 6', 'RUE LAREVELLIERE', '4650', '

## 2. Modeles utilises

Comparaison entre une Regression Lineaire et un Random Forest.
Le Random Forest est retenu comme modele principal (meilleur R²).

In [4]:
with open(config.MODEL_OBJ1, 'rb') as f:
    model = pickle.load(f)

print(f"Modele : {type(model).__name__}")
print(f"Features utilisees : {list(model.feature_names_in_)}")
print(f"Nombre d'arbres : {model.n_estimators}")

Modele : RandomForestRegressor
Features utilisees : ['surface_reelle_bati', 'nombre_pieces_principales', 'surface_terrain', 'latitude', 'longitude', 'type_encode', 'annee', 'mois']
Nombre d'arbres : 100


## 3. Performance du modele

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

FEATURES = list(model.feature_names_in_)
X = df[FEATURES].fillna(0)
y = df[config.COL_PRIX]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
y_pred = model.predict(X_test)

r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R²   : {r2:.4f}")
print(f"MAE  : {mae:,.0f} €")
print(f"RMSE : {rmse:,.0f} €")

R²   : 0.5893
MAE  : 51,916 €
RMSE : 81,641 €


## 4. Predictions 2026

In [6]:
preds = pd.read_csv(config.PREDICTIONS)

resume = preds.groupby(config.COL_TYPE).agg(
    prix_predit_moyen = ('prix_predit', 'mean'),
    prix_min          = ('prix_predit', 'min'),
    prix_max          = ('prix_predit', 'max'),
).round(0)

print("=== Predictions 2026 ===")
print(resume.to_string())

=== Predictions 2026 ===
             prix_predit_moyen  prix_min  prix_max
type_local                                        
Appartement           108319.0   68569.0  181572.0
Maison                216634.0  166463.0  274344.0


## 5. Evolution des prix 2024 → 2025 → 2026